# 3D-printable terrain models

Builds a 6 × 6 inch, 3D-printable terrain model (STL) of a mountain from USGS 3DEP elevation data, plus a hillshade preview image.

Run all cells, then pick a place and a vertical exaggeration in the form at the bottom and click **Build model**.

In [1]:
import numpy as np, tifffile, struct, os, math, time, urllib.request, urllib.error
from PIL import Image
from pyproj import Transformer
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

## Places

Each place is a square area centred on the feature, sized to fit the whole mountain or range. To add your own, add a line with its centre latitude and longitude and the width of the area in km. 3DEP covers the United States and its territories.

In [2]:
# name: (center latitude, center longitude, width of the square area in km)
PLACES = {
    'Mount St Helens': (46.1912, -122.1944, 15),
    'Mt Rainier':      (46.8529, -121.7604, 30),
    'Mt Whitney':      (36.5785, -118.2700, 14),
    'Grand Tetons':    (43.7900, -110.8500, 30),
    'Mt Hood':         (45.3736, -121.6960, 20),
    'Mt Shasta':       (41.4092, -122.1949, 28),
    'Pikes Peak':      (38.8405, -105.0442, 20),
}

## Model parameters

In [3]:
SIZE_MM = 152.4      # model width and depth (6 inches)
GRID = 500           # samples per side (~0.3 mm spacing)
BASE_MM = 3.0        # solid base below lowest point

DEM_DIR = 'dem'          # downloaded DEMs
STL_DIR = 'stl'          # STL files
PREVIEW_DIR = 'preview'  # hillshade previews

## Model functions

The DEM is downloaded in the UTM zone of the place, at 10 m resolution (coarser for areas over 20 km, to keep downloads near 2000 × 2000 px), and cached so rebuilding with a different exaggeration doesn't download it again. The mesh has a top surface, a flat bottom and four side walls, wound so normals point outward. `show_3d` draws an interactive 3D view of the model at its printed proportions, so the exaggeration looks as it will on the print.

In [4]:
def slugify(name):
    return name.lower().replace('.', '').replace(' ', '_')


def place_extent(place):
    """Square UTM extent centred on the place, snapped to the DEM resolution."""
    lat, lon, size_km = PLACES[place]
    size_m = size_km * 1000
    res = max(10.0, math.ceil(size_m / 2000))   # keep downloads near 2000 px per side
    epsg = (32600 if lat >= 0 else 32700) + int(lon // 6 + 31)
    cx, cy = Transformer.from_crs(4326, epsg, always_xy=True).transform(lon, lat)
    xmin = math.floor((cx - size_m / 2) / res) * res
    ymin = math.floor((cy - size_m / 2) / res) * res
    n = int(round(size_m / res))
    return epsg, res, xmin, ymin, n


def download_dem(place):
    """Download the DEM from USGS 3DEP, or reuse a cached copy."""
    epsg, res, xmin, ymin, n = place_extent(place)
    lat, lon, size_km = PLACES[place]
    os.makedirs(DEM_DIR, exist_ok=True)
    dem_file = os.path.join(DEM_DIR, f'dem_{slugify(place)}_{lat:.4f}_{lon:.4f}_{size_km:g}km_{res:g}m.tif')
    try:
        dem = tifffile.imread(dem_file)
    except Exception:   # not downloaded yet, or a bad file from an earlier failed download
        xmax, ymax = xmin + n * res, ymin + n * res
        url = ('https://elevation.nationalmap.gov/arcgis/rest/services/3DEPElevation/ImageServer/exportImage'
               f'?bbox={xmin:.0f},{ymin:.0f},{xmax:.0f},{ymax:.0f}&bboxSR={epsg}&imageSR={epsg}&size={n},{n}'
               '&format=tiff&pixelType=F32&interpolation=RSP_BilinearInterpolation&f=image')
        print(f'Downloading {n} x {n} px DEM at {res:g} m...')
        tmp = dem_file + '.part'
        for attempt in range(5):   # the 3DEP ImageServer sometimes returns 504s or bad data
            try:
                urllib.request.urlretrieve(url, tmp)
                dem = tifffile.imread(tmp)      # rejects JSON error replies and truncated files
                os.replace(tmp, dem_file)
                break
            except Exception as e:
                if os.path.exists(tmp):
                    os.remove(tmp)
                if attempt == 4:
                    raise
                print(f'download failed ({e}), retrying...')
                time.sleep(10 * (attempt + 1))
    dem = np.where(dem.astype(np.float64) < -1000, np.nan, dem.astype(np.float64))
    valid = np.isfinite(dem)
    if not valid.any() or dem[valid].max() - dem[valid].min() < 1:
        raise ValueError(f'No elevation data for {place}: the area may be outside USGS 3DEP coverage '
                         '(for example, over the ocean).')
    return np.where(valid, dem, dem[valid].min()), res


def wall(t_edge, b_edge, flip):
    t0, t1, b0, b1 = t_edge[:-1], t_edge[1:], b_edge[:-1], b_edge[1:]
    if flip:
        return np.concatenate([np.stack([t0, t1, b0], 1), np.stack([t1, b1, b0], 1)])
    return np.concatenate([np.stack([t0, b0, t1], 1), np.stack([t1, b0, b1], 1)])


def write_stl(dem, size_m, exag, path, title):
    """Turn a DEM into a watertight binary STL with a flat base."""
    img = Image.fromarray(dem.astype(np.float32)).resize((GRID, GRID), Image.BILINEAR)
    z_m = np.array(img, dtype=np.float64)
    scale = SIZE_MM / size_m                   # mm per meter horizontally
    z = (z_m - z_m.min()) * scale * exag + BASE_MM
    xs = np.linspace(0, SIZE_MM, GRID)
    ys = np.linspace(SIZE_MM, 0, GRID)         # row 0 = north
    X, Y = np.meshgrid(xs, ys)
    top = np.dstack([X, Y, z])
    bot = np.dstack([X, Y, np.zeros_like(z)])

    tris = []
    # top surface (normal up): rows go north->south, so wind accordingly
    a = top[:-1, :-1]; b = top[:-1, 1:]; c = top[1:, :-1]; d = top[1:, 1:]
    tris.append(np.stack([a, c, b], -2).reshape(-1, 3, 3))
    tris.append(np.stack([b, c, d], -2).reshape(-1, 3, 3))
    # bottom (normal down)
    a = bot[:-1, :-1]; b = bot[:-1, 1:]; c = bot[1:, :-1]; d = bot[1:, 1:]
    tris.append(np.stack([a, b, c], -2).reshape(-1, 3, 3))
    tris.append(np.stack([b, d, c], -2).reshape(-1, 3, 3))
    # side walls
    tris.append(wall(top[0], bot[0], True))          # north edge (x increasing)
    tris.append(wall(top[-1], bot[-1], False))       # south edge
    tris.append(wall(top[:, 0], bot[:, 0], False))   # west edge (y decreasing)
    tris.append(wall(top[:, -1], bot[:, -1], True))  # east edge
    T = np.concatenate(tris).astype(np.float32)

    n = np.cross(T[:, 1] - T[:, 0], T[:, 2] - T[:, 0])
    n /= np.linalg.norm(n, axis=1, keepdims=True) + 1e-12
    rec = np.zeros(len(T), dtype=[('n', '<f4', 3), ('v', '<f4', (3, 3)), ('a', '<u2')])
    rec['n'] = n; rec['v'] = T
    with open(path, 'wb') as f:
        f.write(title.encode('ascii', 'replace')[:80].ljust(80, b' '))
        f.write(struct.pack('<I', len(T)))
        rec.tofile(f)

    # sanity: signed volume positive => outward normals
    vol = np.einsum('ij,ij->i', T[:, 0], np.cross(T[:, 1], T[:, 2])).sum() / 6
    print(f'triangles={len(T)} size={SIZE_MM:.1f}x{SIZE_MM:.1f}mm height={z.max():.1f}mm volume={vol/1000:.1f}cm3')
    return z


def hillshade(dem, res, path):
    gy, gx = np.gradient(dem, res)
    slope = np.arctan(np.hypot(gx, gy)); aspect = np.arctan2(-gx, gy)
    az, alt = np.radians(315), np.radians(45)
    hs = np.sin(alt)*np.cos(slope) + np.cos(alt)*np.sin(slope)*np.cos(az - aspect)
    Image.fromarray((np.clip(hs, 0, 1)*255).astype(np.uint8)).save(path)


TERRAIN_COLORS = [[0.0, '#4a7a46'], [0.35, '#9a9a5a'], [0.6, '#8c6d4f'], [0.85, '#b8a898'], [1.0, '#ffffff']]


def show_3d(z, title, step=2):
    """Interactive 3D view of the model surface, in mm at printed proportions."""
    zs = z[::step, ::step]
    xs = np.linspace(0, SIZE_MM, zs.shape[1])
    ys = np.linspace(SIZE_MM, 0, zs.shape[0])
    fig = go.Figure(go.Surface(x=xs, y=ys, z=zs, colorscale=TERRAIN_COLORS, showscale=False,
                               lighting=dict(ambient=0.5, diffuse=0.8, roughness=0.9, specular=0.1),
                               hovertemplate='x %{x:.0f} mm<br>y %{y:.0f} mm<br>z %{z:.1f} mm<extra></extra>'))
    fig.update_layout(title=title, width=700, height=600, margin=dict(l=30, r=10, t=40, b=10),
                      scene=dict(aspectmode='data', xaxis_title='x (mm)', yaxis_title='y (mm)',
                                 zaxis_title='z (mm)', camera=dict(eye=dict(x=-1.25, y=-1.6, z=0.95), center=dict(x=0, y=0, z=-0.3))))
    fig.show()


def make_model(place, exag):
    """Download the DEM for a place and write its STL and hillshade preview."""
    name = f'{slugify(place)}_{exag:g}x'
    os.makedirs(STL_DIR, exist_ok=True)
    os.makedirs(PREVIEW_DIR, exist_ok=True)
    dem, res = download_dem(place)
    print(f'{place}: {PLACES[place][2]:g} x {PLACES[place][2]:g} km, '
          f'elevation {dem.min():.0f}-{dem.max():.0f} m, exaggeration {exag:g}x')
    z = write_stl(dem, PLACES[place][2] * 1000, exag, os.path.join(STL_DIR, f'{name}.stl'),
                  f'{place} {exag:g}x terrain - USGS 3DEP')
    hillshade(dem, res, os.path.join(PREVIEW_DIR, f'preview_{name}.png'))
    print(f'Saved {STL_DIR}/{name}.stl and {PREVIEW_DIR}/preview_{name}.png')
    return name, z

## Build a model

Choose a place and vertical exaggeration (1.0 = true scale), then click **Build model**. The form then shows the hillshade preview and a 3D view of the model that you can rotate (drag), zoom (scroll) and pan (right-drag). The STL is saved in the `stl` folder as `<place>_<exaggeration>x.stl`, for example `stl/mt_rainier_2x.stl`, and the preview in the `preview` folder. Downloaded DEMs are kept in the `dem` folder.

In [6]:
place_menu = widgets.Dropdown(options=list(PLACES), value='Mount St Helens', description='Place')
exag_slider = widgets.FloatSlider(value=2.0, min=1.0, max=4.0, step=0.25, description='Exaggeration',
                                  readout_format='.2f', style={'description_width': 'initial'})
area_label = widgets.HTML()
build_button = widgets.Button(description='Build model', button_style='primary', icon='cube')
output = widgets.Output()

def show_area(change=None):
    lat, lon, size_km = PLACES[place_menu.value]
    area_label.value = f'{size_km:g} x {size_km:g} km area centred on {lat:.4f}, {lon:.4f}'

def on_build(button):
    build_button.disabled = True
    output.clear_output()
    with output:
        try:
            name, z = make_model(place_menu.value, exag_slider.value)
            display(Image.open(os.path.join(PREVIEW_DIR, f'preview_{name}.png')).resize((500, 500)))
            show_3d(z, f'{place_menu.value}, {exag_slider.value:g}x exaggeration')
        finally:
            build_button.disabled = False

place_menu.observe(show_area, names='value')
build_button.on_click(on_build)
show_area()
display(widgets.VBox([place_menu, area_label, exag_slider, build_button, output]))